# 2.1 Нейронные компьютеры

Задание: **Вариант 3. Квантование весов для NPU**

Реализуйте функцию квантования весов нейронной сети с плавающей точкой (float32) в формат int8 для использования в NPU. Создайте класс QuantizedLayer, который хранит квантованные веса и выполняет прямое распространение с деквантованием. Сравните точность до и после квантования на простых данных.

---

In [1]:
import numpy as np

class FloatLayer:
    def __init__(self, in_features, out_features):
        self.W = np.random.randn(out_features, in_features).astype(np.float32)
        self.b = np.random.randn(out_features).astype(np.float32)

    def forward(self, x):
        return x @ self.W.T + self.b

class QuantizedLayer:
    def __init__(self, float_layer):
        W = float_layer.W

        self.scale = np.max(np.abs(W))
        self.W_q = np.round(W / self.scale).astype(np.int8)

        self.b = float_layer.b.astype(np.float32)
    
    def forward(self, x):
        W_dequant = self.W_q.astype(np.float32) * self.scale
        return x @ W_dequant.T + self.b


np.random.seed(42)

in_features = 4
out_features = 3
batch_size = 10

X = np.random.randn(batch_size, in_features).astype(np.float32)

floar_layer = FloatLayer(in_features, out_features)
float_output = floar_layer.forward(X)

quant_layer = QuantizedLayer(floar_layer)
quant_output = quant_layer.forward(X)

mse = np.mean((float_output - quant_output) ** 2)

print('Float output:', float_output)
print('\nQuantized output:', quant_output)
print('\nMSE:', mse)

Float output: [[-0.8673034   1.2884805   1.0688589 ]
 [-1.3036708   1.210245    1.5796051 ]
 [-0.7368092   0.63638055 -0.05771756]
 [-0.4573259   1.8313508   4.1448407 ]
 [-0.8407507   0.8082536   0.37851852]
 [ 0.78790706 -2.930033    2.503206  ]
 [-1.0399355   2.2640622   0.13068545]
 [-1.6586021   3.9449615   0.4305939 ]
 [-0.5956728  -0.27645773  3.6278472 ]
 [-0.76418173  2.5334444   4.0514946 ]]

Quantized output: [[-0.676922    2.421112    1.274765  ]
 [-0.676922    2.3775163   1.4437923 ]
 [-0.676922    0.6182782   0.07444441]
 [-0.676922   -0.80624837  4.4041896 ]
 [-0.676922   -0.09260994  0.47696882]
 [-0.676922   -4.48421     1.4290522 ]
 [-0.676922    2.2338157   0.83543855]
 [-0.676922    4.9362674   1.5452673 ]
 [-0.676922   -1.5169241   2.8957863 ]
 [-0.676922    0.5905156   4.4859767 ]]

MSE: 0.8887681
